# Clase 4: Introduccion a Naive Bayes
## Un modelo simple, rapido y sorprendentemente efectivo

---

### ¿Que vamos a aprender?

En este notebook aprenderemos:

1. **¿Que es Naive Bayes?** - La intuicion detras del modelo
2. **¿Por que se llama "Naive" (ingenuo)?** - Su supuesto principal
3. **¿Cuando usarlo?** - Escenarios donde brilla y donde no
4. **Ejemplo practico** - Aplicado al problema de churn

---

## 1. La intuicion: Pensemos como detectives

Imaginemos que eres un detective y necesitas determinar si un cliente se va a ir de la empresa (churn). Tienes **pistas** (las variables):

- ¿Lleva poco tiempo como cliente?
- ¿Tiene contrato mes a mes?
- ¿Paga mucho por el servicio?
- ¿Ha llamado muchas veces a soporte?

Tu cerebro hace algo natural: **combina todas las pistas** para tomar una decision. Naive Bayes hace exactamente lo mismo, pero usando **probabilidades**.

### La pregunta que responde Naive Bayes:

> Dadas las caracteristicas de este cliente, ¿cual es la **probabilidad** de que haga churn?

---

## 2. El Teorema de Bayes (simplificado)

El modelo se basa en el **Teorema de Bayes**, que en palabras simples dice:

$$P(\text{Churn} | \text{datos del cliente}) = \frac{P(\text{datos del cliente} | \text{Churn}) \times P(\text{Churn})}{P(\text{datos del cliente})}$$

Traducido a español:

| Termino | Significado |
|---------|-------------|
| $P(\text{Churn} \| \text{datos})$ | Probabilidad de churn **dado lo que sabemos del cliente** (lo que queremos calcular) |
| $P(\text{datos} \| \text{Churn})$ | ¿Que tan comunes son estas caracteristicas **entre los clientes que se fueron**? |
| $P(\text{Churn})$ | ¿Que porcentaje de clientes se va en general? (probabilidad base) |
| $P(\text{datos})$ | ¿Que tan comunes son estas caracteristicas en general? |

### Ejemplo cotidiano

Piensa en el clima: si ves nubes oscuras y humedad alta, ¿cual es la probabilidad de que llueva?

- Sabes que cuando llueve, generalmente hay nubes oscuras (90% del tiempo)
- Sabes que llueve el 30% de los dias
- Sabes que hay nubes oscuras el 40% de los dias

$$P(\text{lluvia} | \text{nubes oscuras}) = \frac{0.90 \times 0.30}{0.40} = 0.675 = 67.5\%$$

Naive Bayes hace este calculo para cada cliente usando sus datos.

---
## 3. ¿Por que se llama "Naive" (ingenuo)?

El modelo asume que **todas las variables son independientes entre si**. Es decir, asume que:

- El tipo de contrato NO influye en el gasto mensual
- La antiguedad NO influye en si tiene soporte tecnico
- El genero NO influye en el metodo de pago

En la vida real, esto **no siempre es cierto** (por ejemplo, los clientes nuevos suelen tener contratos mes a mes). Por eso es "ingenuo".

### Pero... ¿funciona?

**Si, sorprendentemente bien.** A pesar de esta simplificacion, Naive Bayes produce buenos resultados en muchos casos reales. Es como un estudiante que simplifica demasiado un problema pero aun asi llega a una respuesta razonable.

---

## 4. Tipos de Naive Bayes

Existen diferentes versiones segun el tipo de datos:

| Tipo | ¿Cuando usarlo? | Ejemplo |
|------|------------------|---------|
| **GaussianNB** | Variables numericas continuas | Edad, gasto mensual, antiguedad |
| **MultinomialNB** | Conteos o frecuencias | Clasificacion de texto (cantidad de palabras) |
| **BernoulliNB** | Variables binarias (si/no) | Tiene internet si/no, tiene soporte si/no |

En este curso usaremos **GaussianNB** porque nuestros datos tienen variables numericas.

---
## 5. ¿Cuando es bueno usar Naive Bayes?

### Escenarios donde Naive Bayes **brilla**:

| Escenario | ¿Por que? |
|-----------|----------|
| **Clasificacion de texto** (spam, sentimiento, categorias) | Funciona excelente con muchas variables de texto |
| **Datasets pequenos** | Necesita pocos datos para aprender, no sobreajusta facilmente |
| **Muchas variables (alta dimensionalidad)** | Maneja bien datasets con muchas columnas |
| **Necesitas un modelo rapido** | Entrena y predice en milisegundos |
| **Clasificacion en tiempo real** | Ideal para sistemas que necesitan respuestas instantaneas |
| **Modelo base (baseline)** | Perfecto como primer modelo para comparar con otros |

### Escenarios donde Naive Bayes **NO es la mejor opcion**:

| Escenario | ¿Por que? |
|-----------|----------|
| **Variables muy correlacionadas** | El supuesto de independencia falla mucho |
| **Relaciones complejas entre variables** | No captura interacciones (A y B juntas causan C) |
| **Necesitas la mayor precision posible** | Modelos como Random Forest o Gradient Boosting suelen ser superiores |
| **Datos continuos con distribuciones raras** | GaussianNB asume que los datos siguen una curva normal |

### Ejemplos del mundo real:

- **Gmail** usa Naive Bayes para filtrar spam
- **Sistemas de recomendacion** lo usan para clasificar contenido
- **Diagnostico medico** como modelo inicial de triaje
- **Analisis de sentimiento** en redes sociales

---
## 6. Ejemplo practico: Predecir churn con Naive Bayes

Vamos a entrenar un modelo Naive Bayes paso a paso usando el dataset de IBM Telco Customer Churn.

In [ ]:
# Importar librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

sns.set_style("whitegrid")
np.random.seed(42)

print("Librerias importadas correctamente")

In [ ]:
# Cargar y limpiar el dataset
df = pd.read_csv('Telco-Customer-Churn.csv')

# Limpieza basica
df = df.drop('customerID', axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"Dataset: {len(df)} clientes")
print(f"Churn: {df['Churn'].mean()*100:.1f}%")

### Paso 1: Preparar los datos

Naive Bayes necesita datos numericos y, como usa probabilidades basadas en la distribucion normal (campana de Gauss), es importante **escalar** las variables para que todas esten en la misma escala.

In [ ]:
# Codificar variables categoricas
df_encoded = pd.get_dummies(df, columns=df.select_dtypes(include='object').columns, drop_first=True)

# Separar X e y
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar los datos
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Entrenamiento: {X_train.shape[0]} registros")
print(f"Prueba: {X_test.shape[0]} registros")
print(f"Variables: {X_train.shape[1]}")

### Paso 2: Entrenar el modelo

Entrenar un Naive Bayes es **extremadamente rapido**. A diferencia de otros modelos que necesitan iterar muchas veces sobre los datos, Naive Bayes solo necesita **una pasada** para calcular las probabilidades.

In [ ]:
import time

# Entrenar Naive Bayes y medir el tiempo
inicio = time.time()
modelo_nb = GaussianNB()
modelo_nb.fit(X_train, y_train)
tiempo_entrenamiento = time.time() - inicio

print(f"Modelo entrenado en {tiempo_entrenamiento*1000:.2f} milisegundos")
print(f"\nClases aprendidas: {modelo_nb.classes_}")
print(f"Probabilidad base de cada clase:")
print(f"  No Churn (0): {modelo_nb.class_prior_[0]*100:.1f}%")
print(f"  Churn (1):    {modelo_nb.class_prior_[1]*100:.1f}%")

**Observa lo rapido que fue.** Esto es una de las grandes ventajas de Naive Bayes.

Las **probabilidades base** (`class_prior_`) son simplemente el porcentaje de cada clase en los datos de entrenamiento. El modelo las usa como punto de partida antes de ver las caracteristicas del cliente.

### Paso 3: Hacer predicciones y evaluar

In [ ]:
# Hacer predicciones
y_pred = modelo_nb.predict(X_test)

# Evaluar
print("RESULTADOS DE NAIVE BAYES")
print("=" * 40)
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'], zero_division=0)}")

In [ ]:
# Visualizar la matriz de confusion
fig, ax = plt.subplots(figsize=(7, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
ax.set_title('Confusion Matrix - Naive Bayes')
ax.set_ylabel('Valor Real')
ax.set_xlabel('Prediccion')

plt.tight_layout()
plt.show()

# Interpretar los numeros
tn, fp, fn, tp = cm.ravel()
print(f"\nInterpretacion:")
print(f"  Clientes que NO se fueron y el modelo acerto (TN): {tn}")
print(f"  Clientes que NO se fueron pero el modelo dijo que si (FP): {fp}")
print(f"  Clientes que SI se fueron pero el modelo no los detecto (FN): {fn}")
print(f"  Clientes que SI se fueron y el modelo los detecto (TP): {tp}")

---
## 7. La ventaja secreta de Naive Bayes: Probabilidades

A diferencia de otros modelos que solo dicen "churn" o "no churn", Naive Bayes nos da la **probabilidad exacta**. Esto es muy util en el negocio para **priorizar** acciones.

In [ ]:
# Obtener las probabilidades de cada prediccion
probabilidades = modelo_nb.predict_proba(X_test)

# Crear un DataFrame con los resultados
df_proba = pd.DataFrame({
    'Prob_No_Churn': probabilidades[:, 0],
    'Prob_Churn': probabilidades[:, 1],
    'Prediccion': y_pred,
    'Valor_Real': y_test.values
})

print("Primeros 10 clientes con sus probabilidades:")
print("(El modelo predice Churn cuando Prob_Churn > 0.5)")
df_proba.head(10).round(4)

In [ ]:
# Los 10 clientes con MAYOR probabilidad de churn
print("TOP 10 clientes con mayor riesgo de churn:")
print("(Estos son los que la empresa deberia contactar primero)")
print()
df_proba.nlargest(10, 'Prob_Churn')[['Prob_Churn', 'Valor_Real']].round(4)

In [ ]:
# Distribucion de probabilidades de churn
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(df_proba[df_proba['Valor_Real'] == 0]['Prob_Churn'],
        bins=30, alpha=0.6, label='No Churn (real)', color='steelblue')
ax.hist(df_proba[df_proba['Valor_Real'] == 1]['Prob_Churn'],
        bins=30, alpha=0.6, label='Churn (real)', color='salmon')
ax.axvline(x=0.5, color='black', linestyle='--', label='Umbral de decision (0.5)')
ax.set_xlabel('Probabilidad de Churn predicha por el modelo')
ax.set_ylabel('Cantidad de clientes')
ax.set_title('Distribucion de probabilidades de Churn - Naive Bayes')
ax.legend()

plt.tight_layout()
plt.show()

print("Si las distribuciones estan bien separadas, el modelo distingue bien.")
print("Si se solapan mucho, al modelo le cuesta diferenciar churn de no churn.")

### ¿Por que esto es util para la empresa?

En lugar de solo saber "este cliente se va o no", podemos:

- **Ordenar clientes por riesgo**: Llamar primero a los de mayor probabilidad
- **Segmentar acciones**: Probabilidad > 80% = descuento grande, 50-80% = llamada de retencion, < 50% = no hacer nada
- **Calcular el riesgo economico**: Multiplicar la probabilidad de churn por el valor del cliente

---

## 8. Comparacion con Logistic Regression: ¿Cual es mas rapido?

In [ ]:
from sklearn.linear_model import LogisticRegression

# Medir tiempo de Naive Bayes
inicio = time.time()
GaussianNB().fit(X_train, y_train)
tiempo_nb = time.time() - inicio

# Medir tiempo de Logistic Regression
inicio = time.time()
LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced').fit(X_train, y_train)
tiempo_lr = time.time() - inicio

print("COMPARACION DE VELOCIDAD")
print("=" * 40)
print(f"  Naive Bayes:         {tiempo_nb*1000:.2f} ms")
print(f"  Logistic Regression: {tiempo_lr*1000:.2f} ms")
print(f"\n  Naive Bayes es {tiempo_lr/tiempo_nb:.1f}x mas rapido")

---
## 9. Resumen: ¿Cuando elegir Naive Bayes?

### Usa Naive Bayes cuando:

- Necesitas un modelo **rapido** como punto de partida
- Tienes **pocos datos** de entrenamiento
- Trabajas con **clasificacion de texto** (spam, sentimiento, categorias)
- Necesitas **probabilidades** para tomar decisiones graduales
- Quieres un **modelo base** para comparar con otros mas complejos

### NO uses Naive Bayes cuando:

- Las variables estan **muy correlacionadas** entre si
- Necesitas la **maxima precision posible** y tienes suficientes datos
- Las relaciones entre variables son **muy complejas**

### Comparacion rapida con otros modelos:

| Caracteristica | Naive Bayes | Logistic Regression | Random Forest |
|----------------|-------------|---------------------|---------------|
| Velocidad de entrenamiento | Muy rapido | Rapido | Lento |
| Datos necesarios | Pocos | Moderados | Muchos |
| Interpretabilidad | Alta | Alta | Baja |
| Precision tipica | Buena | Buena | Muy buena |
| Manejo de texto | Excelente | Bueno | Regular |
| Manejo de correlaciones | Malo | Bueno | Muy bueno |

---
## 10. EJERCICIO PARA LOS ESTUDIANTES

**Ejercicio 1:** El umbral de decision por defecto es 0.5 (si la probabilidad de churn es mayor a 50%, el modelo predice churn). ¿Que pasaria si bajamos el umbral a 0.3? Completa el codigo de abajo y observa como cambian las metricas.

In [ ]:
# Ejercicio 1: Cambiar el umbral de decision
# En lugar de usar model.predict(), usamos predict_proba() con un umbral personalizado

umbral = 0.3  # Prueba con 0.3, 0.4, 0.5, 0.6

# Obtener probabilidades y aplicar el umbral
proba_churn = modelo_nb.predict_proba(X_test)[:, 1]
y_pred_umbral = (proba_churn >= umbral).astype(int)

print(f"Resultados con umbral = {umbral}")
print("=" * 40)
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_umbral):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_umbral, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_umbral, zero_division=0):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred_umbral, zero_division=0):.4f}")

print(f"\n¿Que cambio respecto al umbral por defecto (0.5)?")
print(f"Escribe tu respuesta aqui:")
# 

**Ejercicio 2:** Responde las siguientes preguntas:

1. Si una empresa de telecomunicaciones quiere contactar a los clientes en riesgo, ¿preferiria un umbral alto (0.7) o bajo (0.3)? ¿Por que?

2. ¿Que pasa con el Recall cuando bajas el umbral? ¿Y con la Precision?

3. Menciona un ejemplo de negocio (diferente a churn) donde Naive Bayes seria una buena opcion.

In [ ]:
# Espacio para las respuestas de los estudiantes

# Pregunta 1:
# 

# Pregunta 2:
# 

# Pregunta 3:
# 